In [72]:
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom

from wsidicom import WsiDicom

from idc_index import index

from PIL import Image
import shutil

DATA_DIR = Path("../data")

ANNOTATION_DIR = DATA_DIR / "annotations"
WSI_DIR = DATA_DIR / "wsi"
CELL_DIR = DATA_DIR / "cell_crops)"
CROP_SIZE = 320

In [5]:
annotation_files = sorted(ANNOTATION_DIR.rglob("*.dcm"))
annotations = [
    pydicom.dcmread(path)
    for path in annotation_files
]

print("Loaded:", len(annotations))

Loaded: 113


In [6]:
diagnoses = pd.Series([
    str(getattr(ds, "AdmittingDiagnosesDescription", "MISSING"))
    for ds in annotations
])

print(diagnoses.value_counts())

Chronic Phase Chronic Myeloid Leukemia, BCR-ABL1 Positive           46
B Acute Lymphoblastic Leukemia                                      32
Mixed Phenotype Acute Leukemia, B/Myeloid                            6
Acute Monocytic Leukemia                                             5
Acute Myeloid Leukemia with Maturation                               5
T Acute Lymphoblastic Leukemia                                       5
Blast Phase Chronic Myeloid Leukemia, BCR-ABL1 Positive              4
MISSING                                                              3
Acute Promyelocytic Leukemia with t(15;17)(q24.1;q21.2);PML-RARA     3
Acute Myelomonocytic Leukemia                                        3
Acute Myeloid Leukemia without Maturation                            1
Name: count, dtype: int64


In [7]:
cml_chronic = [
    ds for ds in annotations
    if "Chronic Phase Chronic Myeloid Leukemia"
    in str(getattr(ds, "AdmittingDiagnosesDescription", ""))
]

print("Chronic-phase CML patients:", len(cml_chronic))

Chronic-phase CML patients: 46


In [9]:
patient_ids = [str(ds.PatientID) for ds in cml_chronic]

target_stages = [
    "promyelocyte",
    "neutrophilic_myelocyte",
    "neutrophilic_metamyelocyte",
    "neutrophilic_band",
    "segmented_neutrophil",
]

print("Annotation DICOMs:", len(cml_chronic))
print("Unique patients:", len(set(patient_ids)))

Annotation DICOMs: 46
Unique patients: 46


In [10]:
rows = []

for ds in cml_chronic:
    patient_id = str(ds.PatientID)

    for group in ds.AnnotationGroupSequence:
        stage = str(group.AnnotationGroupLabel)

        if stage not in target_stages:
            continue

        n = int(group.NumberOfAnnotations)

        coords = np.frombuffer(
            group.PointCoordinatesData,
            dtype="<f4"
        ).reshape(n, 4, 2)

        for i, bbox in enumerate(coords):

            x_min = float(bbox[:, 0].min())
            x_max = float(bbox[:, 0].max())
            y_min = float(bbox[:, 1].min())
            y_max = float(bbox[:, 1].max())

            rows.append({
                "patient_id": patient_id,
                "stage": stage,
                "annotation_index": i,
                "x_min": x_min,
                "y_min": y_min,
                "x_max": x_max,
                "y_max": y_max,
                "width": x_max - x_min,
                "height": y_max - y_min,
            })

cells = pd.DataFrame(rows)

print("Total cells:", len(cells))
print()
print(cells["stage"].value_counts())

Total cells: 8340

stage
segmented_neutrophil          2673
neutrophilic_band             1771
promyelocyte                  1510
neutrophilic_myelocyte        1460
neutrophilic_metamyelocyte     926
Name: count, dtype: int64


In [11]:
cells["exceeds_224"] = (
    (cells["width"] > 224) |
    (cells["height"] > 224)
)

print("Cells exceeding 224 px:", cells["exceeds_224"].sum())

Cells exceeding 224 px: 19


In [12]:
wsi_rows = []

for ds in cml_chronic:
    patient_id = str(ds.PatientID)

    referenced_series = ds.ReferencedSeriesSequence[0]
    referenced_instance = referenced_series.ReferencedInstanceSequence[0]

    wsi_rows.append({
        "patient_id": patient_id,
        "study_uid": str(ds.StudyInstanceUID),
        "wsi_series_uid": str(referenced_series.SeriesInstanceUID),
        "wsi_sop_uid": str(referenced_instance.ReferencedSOPInstanceUID),
    })

wsi_refs = pd.DataFrame(wsi_rows)

print("Patients:", len(wsi_refs))
print("Unique WSI series:", wsi_refs["wsi_series_uid"].nunique())
print("Unique WSI instances:", wsi_refs["wsi_sop_uid"].nunique())

wsi_refs.head()

Patients: 46
Unique WSI series: 46
Unique WSI instances: 46


,patient_id,study_uid,wsi_series_uid,wsi_sop_uid
0,02E74F10E0327AD868D138F2B4FDD6F0,1.2.826.0.1.3680043.8.498.43052523198542423692...,1.2.826.0.1.3680043.8.498.80094077060073240269...,1.2.826.0.1.3680043.8.498.93422388789094217588...
1,1679091C5A880FAF6FB5E6087EB1B2DC,1.2.826.0.1.3680043.8.498.94028615398144787803...,1.2.826.0.1.3680043.8.498.23748388543704898221...,1.2.826.0.1.3680043.8.498.29636807424265518403...
2,17E62166FC8586DFA4D1BC0E1742C08B,1.2.826.0.1.3680043.8.498.46413489303463003013...,1.2.826.0.1.3680043.8.498.82692579400187855946...,1.2.826.0.1.3680043.8.498.80225091671086795258...
3,182BE0C5CDCD5072BB1864CDEE4D3D6E,1.2.826.0.1.3680043.8.498.58188791602161266638...,1.2.826.0.1.3680043.8.498.57703030613872825962...,1.2.826.0.1.3680043.8.498.61884968417046867382...
4,19CA14E7EA6328A42E0EB13D585E4C22,1.2.826.0.1.3680043.8.498.98854805905820444659...,1.2.826.0.1.3680043.8.498.55877079253250534126...,1.2.826.0.1.3680043.8.498.72646957766389953586...


In [13]:
print(
    cells["patient_id"].nunique(),
    wsi_refs["patient_id"].nunique()
)

print(
    "Patients missing WSI reference:",
    len(set(cells["patient_id"]) - set(wsi_refs["patient_id"]))
)

46 46
Patients missing WSI reference: 0


In [14]:
client = index.IDCClient()
client.fetch_index("sm_instance_index")

print(client.sm_instance_index.shape)

2026-09-29 14:25:07,670 - Index sm_instance_index already installed, loading from /Users/abishekchozhan/Library/Application Support/idc_index_data/24.2.2/sm_instance_index.parquet


(391932, 15)


In [ ]:
instance_index = client.sm_instance_index

wsi_metadata = instance_index[
    instance_index["SOPInstanceUID"].isin(wsi_refs["wsi_sop_uid"])
].copy()

print("Matched instances:", len(wsi_metadata))
print("Unique SOP UIDs:", wsi_metadata["SOPInstanceUID"].nunique())


Matched instances: 46
Unique SOP UIDs: 46


In [16]:
print(wsi_metadata["ImageType"].value_counts())

print(
    wsi_metadata[
        [
            "SOPInstanceUID",
            "ImageType",
            "TotalPixelMatrixColumns",
            "TotalPixelMatrixRows",
            "PixelSpacing_0",
            "instance_size",
        ]
    ].head()
)

ImageType
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIMARY, VOLUME, NONE]    1
[ORIGINAL, PRIM

In [19]:
image_types = wsi_metadata["ImageType"].apply(tuple)

print(image_types.value_counts())

expected_type = ("ORIGINAL", "PRIMARY", "VOLUME", "NONE")

print(
    "All original volumes:",
    image_types.apply(lambda x: x == expected_type).all()
)

ImageType
(ORIGINAL, PRIMARY, VOLUME, NONE)    46
Name: count, dtype: int64
All original volumes: True


In [20]:
sizes_gb = wsi_metadata["instance_size"] / 1e9

print(sizes_gb.describe())
print(f"\nTotal if all downloaded: {sizes_gb.sum():.1f} GB")
print(f"Largest WSI: {sizes_gb.max():.1f} GB")

count         46.0
mean       7.53882
std       1.970149
min       2.239634
25%        6.52383
50%       7.379983
75%       8.754209
max      12.721397
Name: instance_size, dtype: Float64

Total if all downloaded: 346.8 GB
Largest WSI: 12.7 GB


In [21]:
print(
    wsi_metadata["PixelSpacing_0"]
    .value_counts()
    .sort_index()
)

PixelSpacing_0
0.000111    44
0.000194     2
Name: count, dtype: int64


In [22]:
different_resolution = wsi_metadata[
    wsi_metadata["PixelSpacing_0"] > 0.00015
]

different_resolution[
    ["SOPInstanceUID", "SeriesInstanceUID",
     "PixelSpacing_0", "TotalPixelMatrixColumns",
     "TotalPixelMatrixRows"]
]

,SOPInstanceUID,SeriesInstanceUID,PixelSpacing_0,TotalPixelMatrixColumns,TotalPixelMatrixRows
389504,1.2.826.0.1.3680043.8.498.62966149116254095056...,1.2.826.0.1.3680043.8.498.19212570839483117674...,0.000194,96768,201472
390705,1.2.826.0.1.3680043.8.498.31354517899517676934...,1.2.826.0.1.3680043.8.498.89770349122349479791...,0.000194,111872,183296


In [23]:
different_sops = set(different_resolution["SOPInstanceUID"])

different_patients = wsi_refs[
    wsi_refs["wsi_sop_uid"].isin(different_sops)
]

print(different_patients[["patient_id", "wsi_sop_uid"]])

print("\nCell counts:")
print(
    cells[
        cells["patient_id"].isin(different_patients["patient_id"])
    ]["patient_id"].value_counts()
)

                          patient_id  \
16  67C6A1E7CE56D3D6FA748AB6D9AF3FD7   
22  A1D0C6E83F027327D8461063F4AC58A6   

                                          wsi_sop_uid  
16  1.2.826.0.1.3680043.8.498.31354517899517676934...  
22  1.2.826.0.1.3680043.8.498.62966149116254095056...  

Cell counts:
patient_id
A1D0C6E83F027327D8461063F4AC58A6    488
67C6A1E7CE56D3D6FA748AB6D9AF3FD7    419
Name: count, dtype: int64


In [26]:
test_patient = cells_with_spacing[
    cells_with_spacing["resolution_group"] == "0.111"
]["patient_id"].iloc[0]

test_ref = wsi_refs[
    wsi_refs["patient_id"] == test_patient
].iloc[0]

test_sop_uid = test_ref["wsi_sop_uid"]

print("Patient:", test_patient)
print("WSI SOP UID:", test_sop_uid)
print(
    "Cells:",
    len(cells[cells["patient_id"] == test_patient])
)

Patient: 02E74F10E0327AD868D138F2B4FDD6F0
WSI SOP UID: 1.2.826.0.1.3680043.8.498.93422388789094217588665950184663030754
Cells: 129


In [27]:
test_wsi_metadata = wsi_metadata[
    wsi_metadata["SOPInstanceUID"] == test_sop_uid
]

print(
    test_wsi_metadata[
        [
            "SOPInstanceUID",
            "PixelSpacing_0",
            "TotalPixelMatrixColumns",
            "TotalPixelMatrixRows",
            "instance_size",
            "ImageType",
        ]
    ].to_string(index=False)
)

                                                  SOPInstanceUID  PixelSpacing_0  TotalPixelMatrixColumns  TotalPixelMatrixRows  instance_size                         ImageType
1.2.826.0.1.3680043.8.498.93422388789094217588665950184663030754        0.000111                   186112                447488     9833878506 [ORIGINAL, PRIMARY, VOLUME, NONE]


In [ ]:
download_dir = WSI_DIR / test_patient
download_dir.mkdir(parents=True, exist_ok=True)

client.download_from_selection(
    downloadDir=str(download_dir),
    sopInstanceUID=[test_sop_uid],
)

2026-09-30 13:21:46,938 - Disk size needed: 9.83 GB
2026-09-30 13:21:46,940 - Disk size available: 110.65 GB
2026-09-30 13:21:54,164 - Not using s5cmd sync as the destination folder is empty or sync or progress bar is not requested
2026-09-30 13:21:54,195 - Initial size of the directory: 0 bytes
2026-09-30 13:21:54,196 - Approximate size of the files that need to be downloaded: 9.83 GB
2026-09-30 13:27:34,083 - Successfully downloaded files to /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/02E74F10E0327AD868D138F2B4FDD6F0


In [ ]:
patient_id = "02E74F10E0327AD868D138F2B4FDD6F0"

output_root = Path("../data/crops") / patient_id
output_root.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. READ CONSENSUS ANNOTATION DICOM
# ============================================================

ann = pydicom.dcmread(ann_files[0])

# ============================================================
# 2. DECODE ALL RECTANGLE ANNOTATIONS
# ============================================================

rows = []
annotation_index = 0

for group in ann.AnnotationGroupSequence:

    label = str(group.AnnotationGroupLabel)
    n = int(group.NumberOfAnnotations)

    # DICOM "OF" data = raw float32 coordinates
    points = np.frombuffer(
        group.PointCoordinatesData,
        dtype="<f4"
    )

    expected = n * 8

    if len(points) != expected:
        print(
            f"WARNING: {label}: "
            f"expected {expected} floats, found {len(points)}"
        )
        continue

    # n rectangles × 4 corners × (x,y)
    rectangles = points.reshape(n, 4, 2)

    for rect in rectangles:

        xs = rect[:, 0]
        ys = rect[:, 1]

        x_min = float(xs.min())
        x_max = float(xs.max())
        y_min = float(ys.min())
        y_max = float(ys.max())

        rows.append({
            "annotation_index": annotation_index,
            "label": label,
            "bbox_x_min": x_min,
            "bbox_y_min": y_min,
            "bbox_x_max": x_max,
            "bbox_y_max": y_max,
            "bbox_width": x_max - x_min,
            "bbox_height": y_max - y_min
        })

        annotation_index += 1

boxes = pd.DataFrame(rows)

#print("Annotations decoded:", len(boxes))
#print()
#print(boxes["label"].value_counts())

In [48]:
# ============================================================
# 3. CALCULATE CROP COORDINATES
# ============================================================

half = CROP_SIZE // 2

metadata = []

for _, row in boxes.iterrows():

    # Center of expert annotation box
    cx = int(round(
        (row["bbox_x_min"] + row["bbox_x_max"]) / 2
    ))

    cy = int(round(
        (row["bbox_y_min"] + row["bbox_y_max"]) / 2
    ))

    # Fixed 320x320 field centered on annotation
    crop_x = cx - half
    crop_y = cy - half

    # Prevent crop from extending beyond slide
    crop_x = max(
        0,
        min(crop_x, slide.size.width - CROP_SIZE)
    )

    crop_y = max(
        0,
        min(crop_y, slide.size.height - CROP_SIZE)
    )

    metadata.append({
        **row.to_dict(),
        "patient_id": patient_id,
        "center_x": cx,
        "center_y": cy,
        "crop_x": crop_x,
        "crop_y": crop_y,
        "crop_size": CROP_SIZE
    })

metadata_df = pd.DataFrame(metadata)

print("Crop coordinates created:", len(metadata_df))

Crop coordinates created: 276


In [49]:
# ============================================================
# 4. EXTRACT CROPS FROM WSI
# ============================================================

filenames = []

for _, row in metadata_df.iterrows():

    label = row["label"]
    idx = int(row["annotation_index"])

    label_dir = output_root / label
    label_dir.mkdir(parents=True, exist_ok=True)

    filename = f"{patient_id}_{idx:04d}_{label}.png"
    filepath = label_dir / filename

    crop = slide.read_region(
        location=(
            int(row["crop_x"]),
            int(row["crop_y"])
        ),
        level=0,
        size=(CROP_SIZE, CROP_SIZE)
    )

    crop.save(filepath)

    filenames.append(str(filepath))

metadata_df["filename"] = filenames

# ============================================================
# 5. SAVE METADATA
# ============================================================

metadata_path = output_root / "metadata.csv"

metadata_df.to_csv(
    metadata_path,
    index=False
)

print("\nFinished.")
print("Crops:", len(metadata_df))
print("Metadata:", metadata_path)


Finished.
Crops: 276
Metadata: ../data/crops/02E74F10E0327AD868D138F2B4FDD6F0/metadata.csv


In [50]:
png_files = list(output_root.rglob("*.png"))

print("Annotations:", len(boxes))
print("Metadata rows:", len(metadata_df))
print("PNG files:", len(png_files))
print(
    "Everything matches:",
    len(boxes) == len(metadata_df) == len(png_files)
)

Annotations: 276
Metadata rows: 276
PNG files: 276
Everything matches: True


In [53]:
CP_CML_DIAGNOSIS = (
    "Chronic Phase Chronic Myeloid Leukemia, BCR-ABL1 Positive"
)

cp_cml_patient_ids = sorted([
    patient_id
    for patient_id, diagnosis in diagnoses
    if diagnosis == CP_CML_DIAGNOSIS
])

print("Total CP-CML patients:", len(cp_cml_patient_ids))

for i, patient_id in enumerate(cp_cml_patient_ids, 1):
    print(f"{i:02d}. {patient_id}")

Total CP-CML patients: 46
01. 02E74F10E0327AD868D138F2B4FDD6F0
02. 1679091C5A880FAF6FB5E6087EB1B2DC
03. 17E62166FC8586DFA4D1BC0E1742C08B
04. 182BE0C5CDCD5072BB1864CDEE4D3D6E
05. 19CA14E7EA6328A42E0EB13D585E4C22
06. 1C383CD30B7C298AB50293ADFECB7B18
07. 1F0E3DAD99908345F7439F8FFABDFFC4
08. 33E75FF09DD601BBE69F351039152189
09. 3416A75F4CEA9109507CACD8E2F2AEFC
10. 34173CB38F07F89DDBEBC2AC9128303F
11. 37693CFC748049E45D87B8C7D8B9AACD
12. 3C59DC048E8850243BE8079A5C74D079
13. 45C48CCE2E2D7FBDEA1AFC51C7C6AD26
14. 4E732CED3463D06DE0CA9A15B6153677
15. 6364D3F0F495B6AB9DCF8D3B5C6E0B01
16. 642E92EFB79421734881B53E1E1B18B6
17. 67C6A1E7CE56D3D6FA748AB6D9AF3FD7
18. 6C8349CC7260AE62E3B1396831A8398F
19. 70EFDF2EC9B086079795C442636B55FB
20. 8E296A067A37563370DED05F5A3BF3EC
21. 98F13708210194C475687BE6106A3B84
22. 9BF31C7FF062936A96D3C8BD1F8F2FF3
23. A1D0C6E83F027327D8461063F4AC58A6
24. A5771BCE93E200C36F7CD9DFD0E5DEAA
25. A5BFC9E07964F8DDDEB95FC584CD965D
26. A6BBC91AE73DD21C0533F735470A9CD0
27. A87FF679

In [59]:
client = index.IDCClient()

def download_wsi(patient_id, wsi_root=Path("../data/wsi")):
    """
    Find and download the single SM series for a patient.
    Returns the downloaded WSI path.
    """

    patient_dir = Path(wsi_root) / patient_id
    patient_dir.mkdir(parents=True, exist_ok=True)

    # Find this patient's SM series
    query = f"""
    SELECT PatientID, SeriesInstanceUID, Modality
    FROM index
    WHERE PatientID = '{patient_id}'
      AND Modality = 'SM'
    """

    sm_series = client.sql_query(query)

    if len(sm_series) != 1:
        raise RuntimeError(
            f"{patient_id}: expected 1 SM series, found {len(sm_series)}"
        )

    series_uid = sm_series.iloc[0]["SeriesInstanceUID"]

    print(f"{patient_id}: downloading WSI...")

    client.download_from_selection(
        downloadDir=str(patient_dir),
        seriesInstanceUID=series_uid,
        quiet=False,
        show_progress_bar=True,
    )

    # Locate downloaded WSI
    wsi_files = list(patient_dir.rglob("SM_*"))

    if len(wsi_files) != 1:
        raise RuntimeError(
            f"{patient_id}: expected 1 downloaded SM file, "
            f"found {len(wsi_files)}"
        )

    wsi_path = wsi_files[0]

    print(f"{patient_id}: download complete")
    print("WSI:", wsi_path)

    return wsi_path

In [60]:
patient_id = "1679091C5A880FAF6FB5E6087EB1B2DC"

wsi_path = download_wsi(patient_id)

2026-09-30 15:31:11,082 - Disk size needed: 10.75 GB
2026-09-30 15:31:11,084 - Disk size available: 95.71 GB


1679091C5A880FAF6FB5E6087EB1B2DC: downloading WSI...


2026-09-30 15:31:13,706 - Not using s5cmd sync as the destination folder is empty or sync or progress bar is not requested
2026-09-30 15:31:13,761 - Initial size of the directory: 0 bytes
2026-09-30 15:31:13,763 - Approximate size of the files that need to be downloaded: 10.75 GB

cp s3://idc-open-data/778a44a8-6efb-4fb9-a70e-ec5c47b3555c/9f207d08-4a65-492c-95c3-1d25fb280a5d.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993/9f207d08-4a65-492c-95c3-1d25fb280a5d.dcm
cp s3://idc-open-data/778a44a8-6efb-4fb9-a70e-ec5c47b3555c/c23739c8-5191-46f8-9ac8-980b9f6ec6f3.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993/c23739c8-5191-46f8-9ac8-980b9f6ec6f3.dcm


cp s3://idc-open-data/778a44a8-6efb-4fb9-a70e-ec5c47b3555c/8995dcb0-f5a8-4a63-98af-b8e4e939797a.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993/8995dcb0-f5a8-4a63-98af-b8e4e939797a.dcm


cp s3://idc-open-data/778a44a8-6efb-4fb9-a70e-ec5c47b3555c/c5e739be-ecef-43d4-9a6b-5d6c7d29603e.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993/c5e739be-ecef-43d4-9a6b-5d6c7d29603e.dcm


cp s3://idc-open-data/778a44a8-6efb-4fb9-a70e-ec5c47b3555c/f47261b0-7ea8-44e6-9401-bc25077b3e5d.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993/f47261b0-7ea8-44e6-9401-bc25077b3e5d.dcm


2026-09-30 15:39:13,106 - Successfully downloaded files to /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC


cp s3://idc-open-data/778a44a8-6efb-4fb9-a70e-ec5c47b3555c/545d4142-154e-4e06-97a0-e338e48358d9.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993/545d4142-154e-4e06-97a0-e338e48358d9.dcm
1679091C5A880FAF6FB5E6087EB1B2DC: download complete
WSI: ../data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC/bonemarrowwsi_pediatricleukemia/1679091C5A880FAF6FB5E6087EB1B2DC/1.2.826.0.1.3680043.8.498.94028615398144787803101943356082784452/SM_1.2.826.0.1.3680043.8.498.23748388543704898221120727307640957993


In [61]:
from pathlib import Path
import pydicom

patient_dir = Path("../data/wsi/1679091C5A880FAF6FB5E6087EB1B2DC")

dcm_files = list(patient_dir.rglob("*.dcm"))

print("DICOM files:", len(dcm_files))

for path in dcm_files:
    ds = pydicom.dcmread(path, stop_before_pixels=True)

    print("\n", path.name)
    print("  Modality:", getattr(ds, "Modality", None))
    print("  ImageType:", getattr(ds, "ImageType", None))
    print("  Rows:", getattr(ds, "Rows", None))
    print("  Columns:", getattr(ds, "Columns", None))
    print("  TotalPixelMatrixRows:", getattr(ds, "TotalPixelMatrixRows", None))
    print("  TotalPixelMatrixColumns:", getattr(ds, "TotalPixelMatrixColumns", None))
    print("  NumberOfFrames:", getattr(ds, "NumberOfFrames", None))

DICOM files: 6

 545d4142-154e-4e06-97a0-e338e48358d9.dcm
  Modality: SM
  ImageType: ['ORIGINAL', 'PRIMARY', 'VOLUME', 'NONE']
  Rows: 1024
  Columns: 1024
  TotalPixelMatrixRows: 448256
  TotalPixelMatrixColumns: 189184
  NumberOfFrames: 81030

 8995dcb0-f5a8-4a63-98af-b8e4e939797a.dcm
  Modality: SM
  ImageType: ['DERIVED', 'PRIMARY', 'VOLUME', 'RESAMPLED']
  Rows: 1024
  Columns: 1024
  TotalPixelMatrixRows: 7004
  TotalPixelMatrixColumns: 2956
  NumberOfFrames: 21

 c5e739be-ecef-43d4-9a6b-5d6c7d29603e.dcm
  Modality: SM
  ImageType: ['DERIVED', 'PRIMARY', 'VOLUME', 'RESAMPLED']
  Rows: 1024
  Columns: 1024
  TotalPixelMatrixRows: 28016
  TotalPixelMatrixColumns: 11824
  NumberOfFrames: 336

 c23739c8-5191-46f8-9ac8-980b9f6ec6f3.dcm
  Modality: SM
  ImageType: ['DERIVED', 'PRIMARY', 'VOLUME', 'RESAMPLED']
  Rows: 1024
  Columns: 1024
  TotalPixelMatrixRows: 1751
  TotalPixelMatrixColumns: 739
  NumberOfFrames: 2

 9f207d08-4a65-492c-95c3-1d25fb280a5d.dcm
  Modality: SM
  ImageType

In [ ]:
# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = Path("../data")
ANN_ROOT = DATA_ROOT
WSI_ROOT = DATA_ROOT / "wsi"
CROP_ROOT = DATA_ROOT / "crops"

CROP_SIZE = 320

# Resolution of our validated standard slide.
# We'll compare MPP rather than raw slide dimensions.
STANDARD_MPP_X = 0.11054455446
STANDARD_MPP_Y = 0.11072368421
MPP_TOLERANCE = 0.005

client = index.IDCClient()

def find_consensus_annotation(patient_id):
    matches = []

    for path in ANN_ROOT.rglob("*"):
        if not path.is_file():
            continue

        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True)
        except Exception:
            continue

        if (
            str(getattr(ds, "PatientID", "")) == patient_id
            and getattr(ds, "Modality", None) == "ANN"
            and "Consensus" in str(
                getattr(ds, "SeriesDescription", "")
            )
        ):
            matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"{patient_id}: expected 1 consensus ANN, "
            f"found {len(matches)}"
        )

    return matches[0]


# ============================================================
# 2. DECODE RECTANGLE ANNOTATIONS
# ============================================================

def decode_annotations(ann_path):

    ann = pydicom.dcmread(ann_path)

    rows = []
    annotation_index = 0

    for group in ann.AnnotationGroupSequence:

        if str(group.GraphicType) != "RECTANGLE":
            continue

        label = str(group.AnnotationGroupLabel)
        n = int(group.NumberOfAnnotations)

        points = np.frombuffer(
            group.PointCoordinatesData,
            dtype="<f4"
        )

        expected = n * 8

        if len(points) != expected:
            raise RuntimeError(
                f"{label}: expected {expected} coordinate "
                f"values, found {len(points)}"
            )

        rectangles = points.reshape(n, 4, 2)

        for rect in rectangles:

            xs = rect[:, 0]
            ys = rect[:, 1]

            x_min = float(xs.min())
            x_max = float(xs.max())
            y_min = float(ys.min())
            y_max = float(ys.max())

            rows.append({
                "annotation_index": annotation_index,
                "label": label,

                "bbox_x_min": x_min,
                "bbox_y_min": y_min,
                "bbox_x_max": x_max,
                "bbox_y_max": y_max,

                "bbox_width": x_max - x_min,
                "bbox_height": y_max - y_min,
            })

            annotation_index += 1

    return pd.DataFrame(rows)

def download_wsi(patient_id):

    patient_dir = WSI_ROOT / patient_id
    patient_dir.mkdir(parents=True, exist_ok=True)

    query = f"""
    SELECT PatientID, SeriesInstanceUID, Modality
    FROM index
    WHERE PatientID = '{patient_id}'
      AND Modality = 'SM'
    """

    sm_series = client.sql_query(query)

    if len(sm_series) != 1:
        raise RuntimeError(
            f"{patient_id}: expected 1 SM series, "
            f"found {len(sm_series)}"
        )

    series_uid = sm_series.iloc[0]["SeriesInstanceUID"]

    print(f"{patient_id}: downloading WSI...")

    client.download_from_selection(
        downloadDir=str(patient_dir),
        seriesInstanceUID=series_uid,
        quiet=False,
        show_progress_bar=True,
    )

    dcm_files = list(patient_dir.rglob("*.dcm"))

    if not dcm_files:
        raise RuntimeError(
            f"{patient_id}: no DICOM files downloaded"
        )

    # Identify directory containing the native-resolution
    # WSI pyramid files.
    series_dirs = set(p.parent for p in dcm_files)

    valid_series_dirs = []

    for directory in series_dirs:
        try:
            test_slide = WsiDicom.open(directory)
            _ = test_slide.size
            valid_series_dirs.append(directory)
            test_slide.close()
        except Exception:
            pass

    if len(valid_series_dirs) != 1:
        raise RuntimeError(
            f"{patient_id}: expected 1 readable WSI series "
            f"directory, found {len(valid_series_dirs)}"
        )

    series_dir = valid_series_dirs[0]

    print(f"{patient_id}: download complete")
    print("Series directory:", series_dir)

    return patient_dir, series_dir

def check_resolution(slide):

    mpp_x = float(slide.mpp.width)
    mpp_y = float(slide.mpp.height)

    standard = (
        abs(mpp_x - STANDARD_MPP_X) <= MPP_TOLERANCE
        and
        abs(mpp_y - STANDARD_MPP_Y) <= MPP_TOLERANCE
    )

    return standard, mpp_x, mpp_y

def extract_crops(patient_id, slide, boxes):

    output_root = CROP_ROOT / patient_id
    output_root.mkdir(parents=True, exist_ok=True)

    half = CROP_SIZE // 2

    metadata = []

    for _, row in boxes.iterrows():

        idx = int(row["annotation_index"])
        label = row["label"]

        cx = int(round(
            (row["bbox_x_min"] + row["bbox_x_max"]) / 2
        ))

        cy = int(round(
            (row["bbox_y_min"] + row["bbox_y_max"]) / 2
        ))

        crop_x = cx - half
        crop_y = cy - half

        crop_x = max(
            0,
            min(crop_x, slide.size.width - CROP_SIZE)
        )

        crop_y = max(
            0,
            min(crop_y, slide.size.height - CROP_SIZE)
        )

        crop = slide.read_region(
            location=(crop_x, crop_y),
            level=0,
            size=(CROP_SIZE, CROP_SIZE),
        )

        label_dir = output_root / label
        label_dir.mkdir(parents=True, exist_ok=True)

        # SAME naming convention as patient #1
        filename = (
            f"{patient_id}_{idx:04d}_{label}.png"
        )

        filepath = label_dir / filename

        crop.save(filepath)

        metadata.append({
            **row.to_dict(),

            "patient_id": patient_id,

            "center_x": cx,
            "center_y": cy,

            "crop_x": crop_x,
            "crop_y": crop_y,
            "crop_size": CROP_SIZE,

            "filename": str(filepath),
        })

    metadata_df = pd.DataFrame(metadata)

    metadata_path = output_root / "metadata.csv"
    metadata_df.to_csv(metadata_path, index=False)

    return metadata_df, metadata_path

def verify_extraction(patient_id, boxes, metadata_df):

    output_root = CROP_ROOT / patient_id

    expected = len(boxes)
    metadata_count = len(metadata_df)

    png_files = list(output_root.rglob("*.png"))
    png_count = len(png_files)

    if expected != metadata_count:
        raise RuntimeError(
            f"{patient_id}: annotation count ({expected}) "
            f"!= metadata rows ({metadata_count})"
        )

    if expected != png_count:
        raise RuntimeError(
            f"{patient_id}: annotation count ({expected}) "
            f"!= PNG count ({png_count})"
        )

    # Make sure every PNG is actually readable
    for path in png_files:
        try:
            with Image.open(path) as img:
                img.verify()
        except Exception as e:
            raise RuntimeError(
                f"{patient_id}: invalid crop {path}: {e}"
            )

    metadata_path = output_root / "metadata.csv"

    if not metadata_path.exists():
        raise RuntimeError(
            f"{patient_id}: metadata.csv missing"
        )

    print(f"✓ {expected} annotations")
    print(f"✓ {metadata_count} metadata rows")
    print(f"✓ {png_count} readable crops")
    print("✓ metadata.csv written")

    return True

def delete_wsi(patient_dir):

    patient_dir = Path(patient_dir)

    if not patient_dir.exists():
        raise RuntimeError(
            f"WSI directory does not exist: {patient_dir}"
        )

    shutil.rmtree(patient_dir)

    if patient_dir.exists():
        raise RuntimeError(
            f"Failed to delete {patient_dir}"
        )

    print("✓ downloaded WSI deleted")

def process_patient(patient_id):

    print("=" * 70)
    print("PATIENT:", patient_id)
    print("=" * 70)

    # ---------- annotation ----------
    ann_path = find_consensus_annotation(patient_id)

    boxes = decode_annotations(ann_path)

    print(f"Consensus annotations: {len(boxes)}")

    # ---------- WSI ----------
    patient_dir, series_dir = download_wsi(patient_id)

    slide = None

    try:
        slide = WsiDicom.open(series_dir)

        print("WSI size:", slide.size)
        print("MPP:", slide.mpp)

        # ---------- resolution ----------
        standard, mpp_x, mpp_y = check_resolution(slide)

        if not standard:

            print()
            print("⚠ NONSTANDARD RESOLUTION")
            print(f"MPP: {mpp_x:.6f} × {mpp_y:.6f}")
            print("Skipping cropping.")
            print("WSI retained for separate processing.")

            return {
                "patient_id": patient_id,
                "status": "nonstandard_resolution",
                "mpp_x": mpp_x,
                "mpp_y": mpp_y,
                "wsi_directory": str(patient_dir),
            }

        # ---------- crops ----------
        metadata_df, metadata_path = extract_crops(
            patient_id,
            slide,
            boxes,
        )

        # Close WSI before verification/deletion
        slide.close()
        slide = None

        # ---------- verification ----------
        verify_extraction(
            patient_id,
            boxes,
            metadata_df,
        )

        # ---------- deletion ----------
        delete_wsi(patient_dir)

        print()
        print(f"✓ {patient_id} COMPLETE")

        return {
            "patient_id": patient_id,
            "status": "complete",
            "annotations": len(boxes),
            "crops": len(metadata_df),
            "metadata": str(metadata_path),
        }

    except Exception:
        # Critical safeguard:
        # WSI is NOT deleted if anything above fails.
        print()
        print("✗ PROCESSING FAILED")
        print("WSI retained.")
        raise

    finally:
        if slide is not None:
            slide.close()

In [88]:
results = []

# Patients #1 and #2 already completed
remaining_patients = cp_cml_patient_ids[33:]

for i, patient_id in enumerate(remaining_patients, start=34):

    print("\n")
    print("#" * 80)
    print(f"PATIENT {i}/46")
    print("#" * 80)

    try:
        result = process_patient(patient_id)
        results.append(result)

    except Exception as e:
        print(f"\nFAILED: {patient_id}")
        print(f"Error: {e}")

        results.append({
            "patient_id": patient_id,
            "status": "failed",
            "error": str(e),
        })

        # Continue to next patient rather than stopping entire batch
        continue



################################################################################
PATIENT 34/46
################################################################################
PATIENT: C51CE410C124A10E0DB5E4B97FC2AF39
Consensus annotations: 235


2026-10-01 23:15:19,560 - Disk size needed: 7.3 GB
2026-10-01 23:15:19,561 - Disk size available: 92.35 GB


C51CE410C124A10E0DB5E4B97FC2AF39: downloading WSI...


2026-10-01 23:15:21,167 - Not using s5cmd sync as the destination folder is empty or sync or progress bar is not requested
2026-10-01 23:15:21,201 - Initial size of the directory: 0 bytes
2026-10-01 23:15:21,202 - Approximate size of the files that need to be downloaded: 7.3 GB






cp s3://idc-open-data/b1db2b8a-e4b1-49e3-ae2f-b03090c5642b/f89f5c0e-114c-4562-8164-173689dd1bfb.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/C51CE410C124A10E0DB5E4B97FC2AF39/bonemarrowwsi_pediatricleukemia/C51CE410C124A10E0DB5E4B97FC2AF39/1.2.826.0.1.3680043.8.498.87818963416990763583645675279002387461/SM_1.2.826.0.1.3680043.8.498.68856705342189514660539878741409834977/f89f5c0e-114c-4562-8164-173689dd1bfb.dcm
cp s3://idc-open-data/b1db2b8a-e4b1-49e3-ae2f-b03090c5642b/2d8c96f3-1a6f-4382-8243-5752da91b4b1.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/C51CE410C124A10E0DB5E4B97FC2AF39/bonemarrowwsi_pediatricleukemia/C51CE410C124A10E0DB5E4B97FC2AF39/1.2.826.0.1.3680043.8.498.87818963416990763583645675279002387461/SM_1.2.826.0.1.3680043.8.498.68856705342189514660539878741409834977/2d8c96f3-1a6f-4382-8243-5752da91b4b1.dcm


cp s3://idc-open-data/b1db2b8a-e4b1-49e3-ae2f-b03090c5642b/3af61329-8409-40c3-8807-b7bc8398e686.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/C51CE410C124A10E0DB5E4B97FC2AF39/bonemarrowwsi_pediatricleukemia/C51CE410C124A10E0DB5E4B97FC2AF39/1.2.826.0.1.3680043.8.498.87818963416990763583645675279002387461/SM_1.2.826.0.1.3680043.8.498.68856705342189514660539878741409834977/3af61329-8409-40c3-8807-b7bc8398e686.dcm


... (114 outputs left out)

cp s3://idc-open-data/7b0a3df4-f40e-4d02-b8d6-b6379cd16d67/85edc9d1-df41-4201-84d3-83a86bf1c2d6.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/F7177163C833DFF4B38FC8D2872F1EC6/bonemarrowwsi_pediatricleukemia/F7177163C833DFF4B38FC8D2872F1EC6/1.2.826.0.1.3680043.8.498.32796004339808195676349160117240960750/SM_1.2.826.0.1.3680043.8.498.11363257719279170754597091350157050426/85edc9d1-df41-4201-84d3-83a86bf1c2d6.dcm


cp s3://idc-open-data/7b0a3df4-f40e-4d02-b8d6-b6379cd16d67/028a98c0-3723-41ee-948d-6aa379ef28ea.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/F7177163C833DFF4B38FC8D2872F1EC6/bonemarrowwsi_pediatricleukemia/F7177163C833DFF4B38FC8D2872F1EC6/1.2.826.0.1.3680043.8.498.32796004339808195676349160117240960750/SM_1.2.826.0.1.3680043.8.498.11363257719279170754597091350157050426/028a98c0-3723-41ee-948d-6aa379ef28ea.dcm


2026-10-02 00:11:53,869 - Successfully downloaded files to /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/F7177163C833DFF4B38FC8D2872F1EC6


cp s3://idc-open-data/7b0a3df4-f40e-4d02-b8d6-b6379cd16d67/9822d7e9-12bc-4c34-99ab-2621860146b6.dcm /Users/abishekchozhan/Documents/peds_heme_onc/peds-leukemia/data/wsi/F7177163C833DFF4B38FC8D2872F1EC6/bonemarrowwsi_pediatricleukemia/F7177163C833DFF4B38FC8D2872F1EC6/1.2.826.0.1.3680043.8.498.32796004339808195676349160117240960750/SM_1.2.826.0.1.3680043.8.498.11363257719279170754597091350157050426/9822d7e9-12bc-4c34-99ab-2621860146b6.dcm
F7177163C833DFF4B38FC8D2872F1EC6: download complete
Series directory: ../data/wsi/F7177163C833DFF4B38FC8D2872F1EC6/bonemarrowwsi_pediatricleukemia/F7177163C833DFF4B38FC8D2872F1EC6/1.2.826.0.1.3680043.8.498.32796004339808195676349160117240960750/SM_1.2.826.0.1.3680043.8.498.11363257719279170754597091350157050426
WSI size: Size(width=188416, height=448768)
MPP: SizeMm(width=0.11054455446, height=0.11072368420999999)
✓ 305 annotations
✓ 305 metadata rows
✓ 305 readable crops
✓ metadata.csv written
✓ downloaded WSI deleted

✓ F7177163C833DFF4B38FC8D2872F1EC

In [86]:
cp_cml_patient_ids[33:]

['C51CE410C124A10E0DB5E4B97FC2AF39',
 'C74D97B01EAE257E44AA9D5BADE97BAF',
 'C81E728D9D4C2F636F067F89CC14862C',
 'C82561EC215A6E31807CEEDF3B3BD25E',
 'C9F0F895FB98AB9159F51FD0297E236D',
 'D3D9446802A44259755D38E6D163E820',
 'D645920E395FEDAD7BBBED0ECA3FE2E0',
 'D67D8AB4F4C10BF22AA353E27879133C',
 'D9D4F495E875A2E075A1A4A6E1B9770F',
 'E369853DF766FA44E1ED0FF613F563BD',
 'E4DA3B7FBBCE2345D7772B0674A318D5',
 'ECCBC87E4B5CE2FE28308FD9F2A7BAF3',
 'F7177163C833DFF4B38FC8D2872F1EC6']

In [90]:
import pydicom
from wsidicom import WsiDicom

different_wsi_files = {}

for patient_id in different_patients:
    patient_dir = WSI_DIR / patient_id

    for path in patient_dir.rglob("*.dcm"):
        ds = pydicom.dcmread(path, stop_before_pixels=True)

        image_type = list(getattr(ds, "ImageType", []))

        if image_type == ["ORIGINAL", "PRIMARY", "VOLUME", "NONE"]:
            different_wsi_files[patient_id] = path

            print("\nPatient:", patient_id)
            print("File:", path)
            print("Matrix:",
                  ds.TotalPixelMatrixColumns,
                  "x",
                  ds.TotalPixelMatrixRows)
            print("PixelSpacing:",
                  ds.SharedFunctionalGroupsSequence[0]
                    .PixelMeasuresSequence[0]
                    .PixelSpacing)


Patient: 67C6A1E7CE56D3D6FA748AB6D9AF3FD7
File: ../data/wsi/67C6A1E7CE56D3D6FA748AB6D9AF3FD7/bonemarrowwsi_pediatricleukemia/67C6A1E7CE56D3D6FA748AB6D9AF3FD7/1.2.826.0.1.3680043.8.498.21421294652089791709923294150455964726/SM_1.2.826.0.1.3680043.8.498.89770349122349479791374740127781241220/4e629abf-d9ad-47a7-8b5e-4d39f2c26a46.dcm
Matrix: 111872 x 183296
PixelSpacing: [0.00019447513812, 0.00019447513812]

Patient: A1D0C6E83F027327D8461063F4AC58A6
File: ../data/wsi/A1D0C6E83F027327D8461063F4AC58A6/bonemarrowwsi_pediatricleukemia/A1D0C6E83F027327D8461063F4AC58A6/1.2.826.0.1.3680043.8.498.37689962845386577627578641862732406672/SM_1.2.826.0.1.3680043.8.498.19212570839483117674134243789833677578/cc462182-1aa4-4acd-ba0c-09aa054b7fab.dcm
Matrix: 96768 x 201472
PixelSpacing: [0.00019447513812, 0.00019447513812]


In [91]:
from PIL import Image
from wsidicom import WsiDicom
from pathlib import Path
import pandas as pd
import numpy as np

# Reference scale used for the other crops
REFERENCE_CROP_SIZE = 320
REFERENCE_SPACING_UM = 0.111

# Exact spacing verified from these two WSIs
LOWRES_SPACING_UM = 0.19447513812

# Same physical field of view as a 320x320 crop at 0.111 um/px
TARGET_FOV_UM = REFERENCE_CROP_SIZE * REFERENCE_SPACING_UM
NATIVE_CROP_SIZE = round(TARGET_FOV_UM / LOWRES_SPACING_UM)

print("Target FOV:", TARGET_FOV_UM, "um")
print("Native crop:", NATIVE_CROP_SIZE, "x", NATIVE_CROP_SIZE)
print("Output crop:", REFERENCE_CROP_SIZE, "x", REFERENCE_CROP_SIZE)

Target FOV: 35.52 um
Native crop: 183 x 183
Output crop: 320 x 320


In [99]:
metadata_by_patient = {}

for patient_id in different_patients:

    # Find consensus annotation DICOM
    annotation_ds = next(
        ds for ds in annotations
        if str(ds.PatientID) == patient_id
    )

    wsi_path = different_wsi_files[patient_id]

    # Match existing structure: ../data/crops/PATIENT_ID/
    patient_output_dir = CELL_DIR / patient_id
    patient_output_dir.mkdir(parents=True, exist_ok=True)

    metadata_rows = []

    print(f"\nProcessing {patient_id}")

    with WsiDicom.open(wsi_path) as slide:

        slide_width = slide.size.width
        slide_height = slide.size.height

        # ALL consensus cell types
        for group in annotation_ds.AnnotationGroupSequence:

            label = str(group.AnnotationGroupLabel)
            n = int(group.NumberOfAnnotations)

            coords = np.frombuffer(
                group.PointCoordinatesData,
                dtype="<f4"
            ).reshape(n, 4, 2)

            label_dir = patient_output_dir / label
            label_dir.mkdir(parents=True, exist_ok=True)

            print(f"  {label}: {n}")

            for annotation_index, bbox in enumerate(coords):

                bbox_x_min = float(bbox[:, 0].min())
                bbox_y_min = float(bbox[:, 1].min())
                bbox_x_max = float(bbox[:, 0].max())
                bbox_y_max = float(bbox[:, 1].max())

                bbox_width = bbox_x_max - bbox_x_min
                bbox_height = bbox_y_max - bbox_y_min

                # Match previous integer center convention
                center_x = int(round((bbox_x_min + bbox_x_max) / 2))
                center_y = int(round((bbox_y_min + bbox_y_max) / 2))

                # 183 native pixels gives approximately same physical FOV
                # as 320 pixels on the standard-resolution slides
                crop_x = int(round(center_x - NATIVE_CROP_SIZE / 2))
                crop_y = int(round(center_y - NATIVE_CROP_SIZE / 2))

                out_of_bounds = (
                    crop_x < 0
                    or crop_y < 0
                    or crop_x + NATIVE_CROP_SIZE > slide_width
                    or crop_y + NATIVE_CROP_SIZE > slide_height
                )

                if out_of_bounds:
                    print(
                        f"WARNING: out-of-bounds "
                        f"{label} {annotation_index}"
                    )
                    continue

                crop_native = slide.read_region(
                    location=(crop_x, crop_y),
                    level=0,
                    size=(NATIVE_CROP_SIZE, NATIVE_CROP_SIZE)
                )

                crop_320 = crop_native.resize(
                    (320, 320),
                    resample=Image.Resampling.BICUBIC
                )

                filename = (
                    label_dir /
                    f"{patient_id}_{annotation_index:04d}_{label}.png"
                )

                crop_320.save(filename)

                metadata_rows.append({
                    "annotation_index": annotation_index,
                    "label": label,
                    "bbox_x_min": bbox_x_min,
                    "bbox_y_min": bbox_y_min,
                    "bbox_x_max": bbox_x_max,
                    "bbox_y_max": bbox_y_max,
                    "bbox_width": bbox_width,
                    "bbox_height": bbox_height,
                    "patient_id": patient_id,
                    "center_x": center_x,
                    "center_y": center_y,
                    "crop_x": crop_x,
                    "crop_y": crop_y,
                    "crop_size": 320,
                    "filename": str(filename),
                })

    patient_metadata = pd.DataFrame(
        metadata_rows,
        columns=[
            "annotation_index",
            "label",
            "bbox_x_min",
            "bbox_y_min",
            "bbox_x_max",
            "bbox_y_max",
            "bbox_width",
            "bbox_height",
            "patient_id",
            "center_x",
            "center_y",
            "crop_x",
            "crop_y",
            "crop_size",
            "filename",
        ]
    )

    patient_metadata.to_csv(
        patient_output_dir / "metadata.csv",
        index=False
    )

    metadata_by_patient[patient_id] = patient_metadata

    print(
        f"Finished: {len(patient_metadata)} cells, "
        f"{patient_metadata['label'].nunique()} labels"
    )


Processing 67C6A1E7CE56D3D6FA748AB6D9AF3FD7
  artifact: 10
  basophilic_erythroblast: 1
  basophilic_metamyelocyte: 2
  basophilic_myelocyte: 4
  damaged_cell: 31
  degranulated_neutrophilic_metamyelocyte: 1
  degranulated_neutrophilic_myelocyte: 3
  eosinophilic_band: 10
  eosinophilic_metamyelocyte: 3
  eosinophilic_myelocyte: 21
  giant_platelet: 1
  lymphocyte: 6
  lymphoidocyte: 2
  mitosis: 4
  monocyte: 6
  myelocytic_blast: 1
  myeloid_precursor_cell: 6
  neutrophil_extracellular_trap: 9
  neutrophilic_band: 59
  neutrophilic_metamyelocyte: 38
  neutrophilic_myelocyte: 130
  no_consensus_found: 18
  orthochromatic_erythroblast: 4
  polychromatic_erythroblast: 2
  promonocyte: 1
  promyelocyte: 60
  segmented_basophil: 13
  segmented_eosinophil: 19
  segmented_neutrophil: 132
  smudge_cell: 6
  technically_unfit: 25
  thrombocyte: 1
  thrombocyte_aggregate: 1
Finished: 630 cells, 33 labels

Processing A1D0C6E83F027327D8461063F4AC58A6
  artifact: 37
  basophilic_erythroblast: 3


In [101]:
audit_rows = []

for ds in cml_chronic:

    patient_id = str(ds.PatientID)
    patient_dir = CELL_DIR / patient_id
    metadata_path = patient_dir / "metadata.csv"

    # Expected counts from consensus DICOM
    expected = {
        str(group.AnnotationGroupLabel): int(group.NumberOfAnnotations)
        for group in ds.AnnotationGroupSequence
    }

    # Metadata
    if metadata_path.exists():
        metadata = pd.read_csv(metadata_path)
        metadata_counts = metadata["label"].value_counts().to_dict()
        duplicate_filenames = metadata["filename"].duplicated().sum()
    else:
        metadata = None
        metadata_counts = {}
        duplicate_filenames = 0

    # PNGs inside label folders
    png_counts = {}

    if patient_dir.exists():
        for label_dir in patient_dir.iterdir():
            if label_dir.is_dir():
                png_counts[label_dir.name] = len(
                    list(label_dir.glob("*.png"))
                )

    # Union catches unexpected labels too
    all_labels = set(expected) | set(metadata_counts) | set(png_counts)

    for label in all_labels:
        audit_rows.append({
            "patient_id": patient_id,
            "label": label,
            "dicom_count": expected.get(label, 0),
            "metadata_count": metadata_counts.get(label, 0),
            "png_count": png_counts.get(label, 0),
            "metadata_exists": metadata_path.exists(),
            "duplicate_filenames": duplicate_filenames,
        })

audit = pd.DataFrame(audit_rows)

audit["match"] = (
    (audit["dicom_count"] == audit["metadata_count"])
    & (audit["dicom_count"] == audit["png_count"])
)

print("Patients checked:", audit["patient_id"].nunique())
print("Patient-label combinations:", len(audit))

print("\nOverall totals:")
print("DICOM annotations:", audit["dicom_count"].sum())
print("Metadata rows:    ", audit["metadata_count"].sum())
print("PNG files:        ", audit["png_count"].sum())

print("\nAll patient/label counts match:", audit["match"].all())

Patients checked: 46
Patient-label combinations: 1309

Overall totals:
DICOM annotations: 14904
Metadata rows:     1412
PNG files:         1412

All patient/label counts match: False


In [103]:
patient_audit = (
    audit.groupby("patient_id")
    .agg(
        dicom_count=("dicom_count", "sum"),
        metadata_count=("metadata_count", "sum"),
        png_count=("png_count", "sum"),
        all_labels_match=("match", "all"),
        metadata_exists=("metadata_exists", "all"),
        duplicate_filenames=("duplicate_filenames", "max"),
    )
    .reset_index()
)

patient_audit["total_match"] = (
    (patient_audit["dicom_count"] == patient_audit["metadata_count"])
    & (patient_audit["dicom_count"] == patient_audit["png_count"])
)

print(patient_audit.to_string(index=False))

                      patient_id  dicom_count  metadata_count  png_count  all_labels_match  metadata_exists  duplicate_filenames  total_match
02E74F10E0327AD868D138F2B4FDD6F0          256               0          0             False            False                    0        False
1679091C5A880FAF6FB5E6087EB1B2DC          317               0          0             False            False                    0        False
17E62166FC8586DFA4D1BC0E1742C08B          430               0          0             False            False                    0        False
182BE0C5CDCD5072BB1864CDEE4D3D6E          316               0          0             False            False                    0        False
19CA14E7EA6328A42E0EB13D585E4C22          348               0          0             False            False                    0        False
1C383CD30B7C298AB50293ADFECB7B18          173               0          0             False            False                    0        False
1F0E3D

In [104]:
problems = audit[~audit["match"]].copy()

print("\nMismatched patient/label combinations:", len(problems))

if len(problems):
    print(
        problems[
            [
                "patient_id",
                "label",
                "dicom_count",
                "metadata_count",
                "png_count",
            ]
        ].sort_values(["patient_id", "label"]).to_string(index=False)
    )
else:
    print("No count mismatches found.")


Mismatched patient/label combinations: 1244
                      patient_id                                   label  dicom_count  metadata_count  png_count
02E74F10E0327AD868D138F2B4FDD6F0                                artifact            1               0          0
02E74F10E0327AD868D138F2B4FDD6F0                    basophilic_myelocyte            1               0          0
02E74F10E0327AD868D138F2B4FDD6F0                            damaged_cell           55               0          0
02E74F10E0327AD868D138F2B4FDD6F0                       eosinophilic_band            3               0          0
02E74F10E0327AD868D138F2B4FDD6F0              eosinophilic_metamyelocyte            2               0          0
02E74F10E0327AD868D138F2B4FDD6F0                  eosinophilic_myelocyte            2               0          0
02E74F10E0327AD868D138F2B4FDD6F0                             erythrocyte            4               0          0
02E74F10E0327AD868D138F2B4FDD6F0                   

In [106]:
CROP_DIR = Path("../data/crops")

In [108]:
patient_audit = (
    audit.groupby("patient_id")
    .agg(
        dicom_count=("dicom_count", "sum"),
        metadata_count=("metadata_count", "sum"),
        png_count=("png_count", "sum"),
        all_labels_match=("match", "all"),
        metadata_exists=("metadata_exists", "all"),
        duplicate_filenames=("duplicate_filenames", "max"),
    )
    .reset_index()
)

patient_audit["total_match"] = (
    (patient_audit["dicom_count"] == patient_audit["metadata_count"])
    & (patient_audit["dicom_count"] == patient_audit["png_count"])
)

print("Patients checked:", len(patient_audit))
print("Complete patients:", patient_audit["total_match"].sum())

print("\nOVERALL TOTALS")
print("DICOM annotations:", patient_audit["dicom_count"].sum())
print("Metadata rows:    ", patient_audit["metadata_count"].sum())
print("PNG files:        ", patient_audit["png_count"].sum())

print("\nAll patients match:", patient_audit["total_match"].all())
print(
    "Duplicate metadata filenames:",
    patient_audit["duplicate_filenames"].sum()
)

Patients checked: 46
Complete patients: 28

OVERALL TOTALS
DICOM annotations: 14904
Metadata rows:     15288
PNG files:         15288

All patients match: False
Duplicate metadata filenames: 0


In [111]:
audit_rows = []

for ds in cml_chronic:

    patient_id = str(ds.PatientID)
    patient_dir = CROP_DIR / patient_id
    metadata_path = patient_dir / "metadata.csv"

    # Sum ALL DICOM annotation groups sharing the same label
    expected = {}

    for group in ds.AnnotationGroupSequence:
        label = str(group.AnnotationGroupLabel)
        count = int(group.NumberOfAnnotations)

        expected[label] = expected.get(label, 0) + count

    # Metadata
    metadata = pd.read_csv(metadata_path)

    metadata_counts = (
        metadata["label"]
        .value_counts()
        .to_dict()
    )

    # PNGs
    png_counts = {}

    for label_dir in patient_dir.iterdir():
        if label_dir.is_dir():
            png_counts[label_dir.name] = len(
                list(label_dir.glob("*.png"))
            )

    all_labels = set(expected) | set(metadata_counts) | set(png_counts)

    for label in all_labels:
        audit_rows.append({
            "patient_id": patient_id,
            "label": label,
            "dicom_count": expected.get(label, 0),
            "metadata_count": metadata_counts.get(label, 0),
            "png_count": png_counts.get(label, 0),
        })

audit = pd.DataFrame(audit_rows)

audit["match"] = (
    (audit["dicom_count"] == audit["metadata_count"])
    & (audit["dicom_count"] == audit["png_count"])
)

print("Patients checked:", audit["patient_id"].nunique())

print("\nOVERALL TOTALS")
print("DICOM annotations:", audit["dicom_count"].sum())
print("Metadata rows:    ", audit["metadata_count"].sum())
print("PNG files:        ", audit["png_count"].sum())

print("\nAll counts match:", audit["match"].all())
print("Mismatches:", (~audit["match"]).sum())

Patients checked: 46

OVERALL TOTALS
DICOM annotations: 15288
Metadata rows:     15288
PNG files:         15288

All counts match: True
Mismatches: 0
